In [20]:
from pathlib import Path
import pandas as pd
import numpy as np
#import h5py

In [23]:
ROOT = Path().cwd()
#MAZUR = Path("/data/mazurovev/all_marks/methylation/lncRNA_Peaks_corrs").resolve()
DATA = Path("../data").resolve()
DATA_INITIAL = Path("../data_initial").resolve()
HIMORNA_INITIAL = DATA_INITIAL / "himorna"
HIMORNA = DATA / "himorna"
METHYL_INITIAL = HIMORNA_INITIAL / "methylation"
METHYL = HIMORNA / "methylation"

## Код для получения файлов из METHYL

Поскольку исходники находтся на сервере himorna в папке MAZUR, то было принято решение не переносить их. Дальнейшая работа основывается уже на tsv.gz файлах.

def get_all_correlated_peaks(hm, to_hdf5, to_save):
    if to_save.exists():
        print(f"Skipped: {to_hdf5.stem}")
        return None
    all_results = []

    # Открываем HDF5 файл
    with h5py.File(to_hdf5, 'r') as f:
        lncRNAs = list(f['lncRNAs_names'][:])
        corrs_matrix = f['corrs_matrix'][:]

    # Читаем пики
    peaks = pd.read_csv(Path("/data/mazurovev/all_marks") / hm / "merged_peaks_first_in_biosample.bed", sep="\t", header=None)

    for i, lncRNA in enumerate(lncRNAs):
        corrs = corrs_matrix[i, :]
        nonzero_indices = np.nonzero(corrs)[0]
        nonzero_corrs = corrs[nonzero_indices]
        nonzero_peaks = ["peak_" + str(j) for j in nonzero_indices]

        # Извлекаем соответствующие пики
        res = peaks[peaks[3].isin(nonzero_peaks)].copy()
        res["corr"] = nonzero_corrs
        res["lnc_ens"] = lncRNA.decode('utf-8')  # Декодируем байтовую строку в строку

        all_results.append(res)

    # Объединяем все результаты в один DataFrame
    final_result = pd.concat(all_results, ignore_index=True)
    
    final_result.to_csv(to_save, sep="\t", compression="gzip", index=False)

for to_hdf5 in MAZUR.glob("*.hdf5"):
    print(to_hdf5.stem)
    to_save = METHYL / f"{to_hdf5.stem}.tsv.gz"
    get_all_correlated_peaks(hm="methylation", to_hdf5=to_hdf5, to_save=to_save)

## Объединение пиков метилирования

In [14]:
to_df = METHYL_INITIAL / "lncRNA_Peaks_Correlations_corrected_non_zero_0.tsv.gz"
df = pd.read_csv(to_df, sep="\t", names=["chrom","start","end","peak","corr","lnc_ens"], header=0)

In [15]:
df

,chrom,start,end,peak,corr,lnc_ens
0,chr12,49123587,49123588,peak_10386212,-0.887848,ENSG00000099869
1,chr8,142304068,142304069,peak_49210615,0.911249,ENSG00000099869
2,chr1,15837335,15837336,peak_558140,-0.870478,ENSG00000103472
3,chr1,66118486,66118487,peak_1734200,0.864452,ENSG00000103472
4,chr1,199066625,199066626,peak_3431110,0.878119,ENSG00000103472
...,...,...,...,...,...,...
26777104,chr5,175979550,175979551,peak_57225158,-0.585292,ENSG00000206573
26777105,chr5,181449969,181449970,peak_57225437,-0.580529,ENSG00000206573
26777106,chr9,67459634,67459635,peak_57236284,-0.569167,ENSG00000206573
26777107,chr11,126362943,126362944,peak_9357347,-0.885380,ENSG00000212766


In [16]:
list_of_meth_tables = []
for to_meth_table in METHYL_INITIAL.glob("*tsv.gz"):
    print(to_meth_table)
    list_of_meth_tables.append(pd.read_csv(to_meth_table, sep="\t", names=["chrom","start","end","peak","corr","lnc_ens"], header=0))
methyl_peaks = pd.concat(list_of_meth_tables)

/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/himorna/methylation/lncRNA_Peaks_Correlations_corrected_non_zero_0.tsv.gz
/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/himorna/methylation/lncRNA_Peaks_Correlations_corrected_non_zero_10.tsv.gz
/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/himorna/methylation/lncRNA_Peaks_Correlations_corrected_non_zero_11.tsv.gz
/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/himorna/methylation/lncRNA_Peaks_Correlations_corrected_non_zero_12.tsv.gz
/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/himorna/methylation/lncRNA_Peaks_Correlations_corrected_non_zero_13.tsv.gz
/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data_initial/himorna/methylation/lncRNA_Peaks_Correlations_corrected_non_zero_14.tsv.gz
/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks/data

In [17]:
methyl_peaks

,chrom,start,end,peak,corr,lnc_ens
0,chr12,49123587,49123588,peak_10386212,-0.887848,ENSG00000099869
1,chr8,142304068,142304069,peak_49210615,0.911249,ENSG00000099869
2,chr1,15837335,15837336,peak_558140,-0.870478,ENSG00000103472
3,chr1,66118486,66118487,peak_1734200,0.864452,ENSG00000103472
4,chr1,199066625,199066626,peak_3431110,0.878119,ENSG00000103472
...,...,...,...,...,...,...
25858161,chr17,81669387,81669388,peak_21156386,-0.897134,ENSG00000261222
25858162,chr2,222626043,222626044,peak_28291839,-0.893882,ENSG00000261222
25858163,chr3,120774402,120774403,peak_33936371,-0.937961,ENSG00000261222
25858164,chr8,9741473,9741474,peak_47069123,-0.889871,ENSG00000261222


In [21]:
methyl_peaks["corr"] = np.round(methyl_peaks["corr"], 3)
methyl_peaks["corr_sign"] = np.where(methyl_peaks["corr"] > 0, "plus", "minus")

In [24]:
methyl_peaks.head()

,chrom,start,end,peak,corr,lnc_ens,corr_sign
0,chr12,49123587,49123588,peak_10386212,-0.888,ENSG00000099869,minus
1,chr8,142304068,142304069,peak_49210615,0.911,ENSG00000099869,plus
2,chr1,15837335,15837336,peak_558140,-0.870,ENSG00000103472,minus
3,chr1,66118486,66118487,peak_1734200,0.864,ENSG00000103472,plus
4,chr1,199066625,199066626,peak_3431110,0.878,ENSG00000103472,plus


In [25]:
to_save =  METHYL / "methylation_peaks.tsv.gz"
methyl_peaks.to_csv(to_save, sep="\t", compression="gzip", index=False)  